In [ ]:
# ============================================================
# RUNTIME SETTINGS
# ============================================================
# Required: CPU (Standard)
# GPU: Not required
# High-RAM: Recommended for large datasets
#
# SETUP: Add GITHUB_TOKEN to Colab Secrets (key icon in sidebar)
# ============================================================

import subprocess
from google.colab import userdata

# Get GitHub token from Colab Secrets (for private repo access)
token = userdata.get("GITHUB_TOKEN")
if not token:
    raise ValueError(
        "GITHUB_TOKEN not found in Colab Secrets.\n"
        "1. Click the key icon in the left sidebar\n"
        "2. Add a secret named 'GITHUB_TOKEN' with your GitHub token\n"
        "3. Toggle 'Notebook access' ON"
    )

# Install package from private GitHub repo
repo_url = f"git+https://{token}@github.com/SilasPignotti/urban-tree-transfer.git"
subprocess.run(["pip", "install", repo_url, "-q"], check=True)

print("OK: Package installed")


In [ ]:
# Mount Google Drive for data files
from google.colab import drive

drive.mount("/content/drive")

print("Google Drive mounted")


In [ ]:
# Package imports
from urban_tree_transfer.config import PROJECT_CRS, RANDOM_SEED
from urban_tree_transfer.config.loader import load_feature_config, get_all_feature_names
from urban_tree_transfer.utils import ExecutionLog, save_figure, setup_plotting
from urban_tree_transfer.utils.plotting import PUBLICATION_STYLE

from pathlib import Path
from datetime import datetime, timezone
import json
import warnings

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from shapely.geometry import box
from scipy.interpolate import interp1d

setup_plotting()
log = ExecutionLog("exp_05_spatial_autocorrelation")

warnings.filterwarnings("ignore", category=UserWarning)

print("OK: Package imports complete")


In [ ]:
# ============================================================
# SPATIAL ANALYSIS DEPENDENCIES
# ============================================================
# Install PySAL/ESDA if not available (not in default environment)

try:
    from pysal.lib import weights
    from esda.moran import Moran
    print("OK: PySAL/ESDA available")
except ImportError:
    print("Installing spatial analysis libraries...")
    subprocess.run(["pip", "install", "pysal", "esda", "-q"], check=True)
    from pysal.lib import weights
    from esda.moran import Moran
    print("OK: PySAL/ESDA installed and imported")


In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

DRIVE_DIR = Path("/content/drive/MyDrive/dev/urban-tree-transfer")
INPUT_DIR = DRIVE_DIR / "data" / "phase_2_quality"
OUTPUT_DIR = DRIVE_DIR / "outputs" / "phase_2"
METADATA_DIR = OUTPUT_DIR / "metadata"
LOGS_DIR = OUTPUT_DIR / "logs"
FIGURES_DIR = OUTPUT_DIR / "figures" / "exp_05_spatial"

CITIES = ["berlin", "leipzig"]
DISTANCE_LAGS = [100, 150, 200, 250, 300, 350, 400, 450, 500, 600, 700, 800, 1000, 1200]
BLOCK_SIZE_CANDIDATES = [200, 300, 400, 500, 600, 800, 1000]

SAMPLE_SIZE = 50_000  # set to None to disable sampling
FORCE_RECOMPUTE = False

for d in [METADATA_DIR, LOGS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

feature_config = load_feature_config()

temporal_path = METADATA_DIR / "temporal_selection.json"
selected_months = [7]
if temporal_path.exists():
    temporal_cfg = json.loads(temporal_path.read_text())
    selected_months = temporal_cfg.get("selected_months", selected_months)

all_feature_names = get_all_feature_names(feature_config, months=selected_months)
all_feature_names.append("CHM_1m")

representative_features = [
    "NDVI_07",
    "CHM_1m",
    "B8_07",
    "B4_07",
    "NDre1_07",
]

print(f"Input (Phase 2):   {INPUT_DIR}")
print(f"Output (JSONs):    {METADATA_DIR}")
print(f"Figures:           {FIGURES_DIR}")
print(f"Logs (Drive):      {LOGS_DIR}")
print(f"Cities:            {CITIES}")
print(f"Distance lags:     {DISTANCE_LAGS}")
print(f"Random seed:       {RANDOM_SEED}")
print(f"Selected months:   {selected_months}")


In [ ]:
# ============================================================
# SECTION A: Moran's I Calculation
# ============================================================

log.start_step("Moran's I Calculation")

results_file = METADATA_DIR / "morans_i_results.parquet"


def resolve_feature_name(feature: str, columns: set[str], preferred_month: int = 7) -> str | None:
    if feature in columns:
        return feature
    if "_" in feature:
        base = feature.rsplit("_", 1)[0]
        preferred = f"{base}_{preferred_month:02d}"
        if preferred in columns:
            return preferred
        matches = sorted(
            [c for c in columns if c.startswith(base + "_") and c[-2:].isdigit()]
        )
        if matches:
            return matches[0]
    return None


def maybe_sample(gdf: gpd.GeoDataFrame, sample_size: int | None) -> gpd.GeoDataFrame:
    if sample_size is None or len(gdf) <= sample_size:
        return gdf
    return gdf.sample(sample_size, random_state=RANDOM_SEED).copy()


if results_file.exists() and not FORCE_RECOMPUTE:
    print(f"Found existing Moran's I results: {results_file}")
    morans_df = pd.read_parquet(results_file)
    print(f"Loaded {len(morans_df):,} cached results")
    log.end_step(status="success", records=len(morans_df))
else:
    results = []
    city_data = {}

    for city in CITIES:
        path = INPUT_DIR / f"trees_clean_{city}.gpkg"
        print(f"Loading: {path}")
        gdf = gpd.read_file(path)

        if gdf.crs is None or gdf.crs.to_epsg() != int(str(PROJECT_CRS).split(":")[-1]):
            raise ValueError(f"Invalid CRS for {city}: {gdf.crs}. Expected {PROJECT_CRS}.")

        gdf = maybe_sample(gdf, SAMPLE_SIZE).reset_index(drop=True)
        city_data[city] = gdf
        print(f"  {city}: {len(gdf):,} rows")

    for city, gdf in city_data.items():
        available_cols = set(gdf.columns)
        resolved_features = []
        for feat in representative_features:
            resolved = resolve_feature_name(feat, available_cols)
            if resolved:
                resolved_features.append(resolved)
            else:
                print(f"Warning: feature not found for {city}: {feat}")

        resolved_features = sorted(set(resolved_features))
        print(f"{city}: using {resolved_features}")

        weights_cache = {}
        for distance in DISTANCE_LAGS:
            w = weights.DistanceBand.from_dataframe(
                gdf,
                threshold=distance,
                silence_warnings=True,
                binary=True,
            )
            w.transform = "R"
            weights_cache[distance] = w

        for feature in resolved_features:
            values = gdf[feature].astype(float)
            if values.var() == 0:
                print(f"Skipping constant feature: {feature} ({city})")
                continue

            for distance, w in weights_cache.items():
                moran = Moran(values.values, w)
                results.append({
                    "city": city,
                    "feature": feature,
                    "distance_m": distance,
                    "morans_i": float(moran.I),
                    "p_value": float(moran.p_sim),
                    "significant": bool(moran.p_sim < 0.05),
                    "n": len(values),
                })

    morans_df = pd.DataFrame(results)

    # Basic validation
    if not ((morans_df["morans_i"] >= -1).all() and (morans_df["morans_i"] <= 1).all()):
        raise ValueError("Moran's I values must be between -1 and 1.")

    morans_df.to_parquet(results_file)
    print(f"Saved Moran's I results: {results_file}")

    log.end_step(status="success", records=len(morans_df))


In [ ]:
# ============================================================
# SECTION B: Autocorrelation Decay Analysis
# ============================================================

log.start_step("Autocorrelation Decay Analysis")


def find_decay_distance(morans_subset: pd.DataFrame, threshold: float = 0.05) -> float:
    sorted_data = morans_subset.sort_values("distance_m")

    if sorted_data.iloc[0]["morans_i"] < threshold:
        return float(sorted_data.iloc[0]["distance_m"])

    for i in range(len(sorted_data) - 1):
        y0 = sorted_data.iloc[i]["morans_i"]
        y1 = sorted_data.iloc[i + 1]["morans_i"]
        if y0 >= threshold and y1 < threshold:
            x = [sorted_data.iloc[i]["distance_m"], sorted_data.iloc[i + 1]["distance_m"]]
            y = [y0, y1]
            f = interp1d(y, x)
            return float(f(threshold))

    return float(sorted_data.iloc[-1]["distance_m"])


decay_rows = []
for city in CITIES:
    city_df = morans_df[morans_df["city"] == city]
    for feature in sorted(city_df["feature"].unique()):
        subset = city_df[city_df["feature"] == feature]
        decay_distance = find_decay_distance(subset, threshold=0.05)
        decay_rows.append({
            "city": city,
            "feature": feature,
            "decay_distance_m": decay_distance,
        })

decay_df = pd.DataFrame(decay_rows)

log.end_step(status="success", records=len(decay_df))


In [ ]:
# ============================================================
# SECTION C: Optimal Block Size Determination
# ============================================================

log.start_step("Optimal Block Size Determination")

# Aggregate decay distances across cities per feature
summary_df = (
    decay_df
    .groupby("feature")
    .agg(
        decay_mean=("decay_distance_m", "mean"),
        decay_std=("decay_distance_m", "std"),
        decay_max=("decay_distance_m", "max"),
    )
    .reset_index()
)

max_decay_distance = summary_df["decay_max"].max()
mean_plus_std = (summary_df["decay_mean"] + summary_df["decay_std"].fillna(0)).max()

safety_buffer_m = 100
recommended_block_size = int(np.ceil((max_decay_distance + safety_buffer_m) / 100) * 100)

proportional_buffer_20pct = int(np.ceil((max_decay_distance * 1.2) / 10) * 10)

print(f"Maximum decay distance: {max_decay_distance:.1f} m")
print(f"Mean + std (max across features): {mean_plus_std:.1f} m")
print(f"Safety buffer: {safety_buffer_m} m")
print(f"Recommended block size: {recommended_block_size} m")
print(f"Proportional (20%) buffer: {proportional_buffer_20pct} m")

log.end_step(status="success", records=len(summary_df))


In [ ]:
# ============================================================
# SECTION D: Cross-City Validation
# ============================================================

log.start_step("Cross-City Validation")

city_decay = decay_df.groupby("city")["decay_distance_m"].max().to_dict()

cross_city_diff = abs(city_decay["berlin"] - city_decay["leipzig"])

if cross_city_diff > 50:
    print(
        f"Warning: Decay distance differs by {cross_city_diff:.1f} m across cities."
    )
else:
    print(f"Cross-city consistency is high (diff {cross_city_diff:.1f} m).")

log.end_step(status="success", records=len(city_decay))


In [ ]:
# ============================================================
# SECTION E: Block Count & Feasibility + Sensitivity Analysis
# ============================================================

log.start_step("Block Feasibility & Sensitivity")


def build_grid(bounds: tuple[float, float, float, float], cell_size: int, crs) -> gpd.GeoDataFrame:
    minx, miny, maxx, maxy = bounds
    xs = np.arange(minx, maxx + cell_size, cell_size)
    ys = np.arange(miny, maxy + cell_size, cell_size)

    polygons = [box(x, y, x + cell_size, y + cell_size) for x in xs for y in ys]
    grid = gpd.GeoDataFrame({"geometry": polygons}, crs=crs)
    grid["block_id"] = np.arange(len(grid))
    return grid


def count_blocks(gdf: gpd.GeoDataFrame, cell_size: int) -> tuple[gpd.GeoDataFrame, pd.Series]:
    boundary = gdf.unary_union.convex_hull
    grid = build_grid(gdf.total_bounds, cell_size, gdf.crs)

    grid = grid[grid.intersects(boundary)].copy()
    grid = grid.reset_index(drop=True)
    grid["block_id"] = np.arange(len(grid))

    joined = gpd.sjoin(gdf[["geometry"]], grid, predicate="within", how="left")
    counts = joined.groupby("block_id").size().reindex(grid["block_id"]).fillna(0)
    grid["tree_count"] = counts.values

    return grid, counts


def interpolate_moran(city_df: pd.DataFrame, feature: str, distance: int) -> float:
    subset = city_df[city_df["feature"] == feature].sort_values("distance_m")
    xs = subset["distance_m"].values
    ys = subset["morans_i"].values
    if distance in xs:
        return float(subset[subset["distance_m"] == distance]["morans_i"].iloc[0])
    f = interp1d(xs, ys, fill_value="extrapolate")
    return float(f(distance))


block_stats = {}
residual_autocorr = {city: [] for city in CITIES}
block_counts = {city: [] for city in CITIES}

for city in CITIES:
    city_gdf = gpd.read_file(INPUT_DIR / f"trees_clean_{city}.gpkg")
    grid, counts = count_blocks(city_gdf, recommended_block_size)

    blocks_with_trees = int((counts > 0).sum())
    avg_trees_per_block = float(city_gdf.shape[0] / max(blocks_with_trees, 1))
    feasible = blocks_with_trees >= 30

    block_stats[city] = {
        "blocks": blocks_with_trees,
        "avg_trees_per_block": avg_trees_per_block,
        "feasible": feasible,
        "grid": grid,
    }

    # Sensitivity analysis
    city_df = morans_df[morans_df["city"] == city]
    features = sorted(city_df["feature"].unique())

    for size in BLOCK_SIZE_CANDIDATES:
        grid_size, counts_size = count_blocks(city_gdf, size)
        blocks_with_trees_size = int((counts_size > 0).sum())
        block_counts[city].append(blocks_with_trees_size)

        feature_vals = [interpolate_moran(city_df, feat, size) for feat in features]
        residual_autocorr[city].append(float(np.mean(feature_vals)))

log.end_step(status="success", records=len(block_stats))


In [ ]:
# ============================================================
# SECTION: Visualizations
# ============================================================

log.start_step("Visualizations")

# Plot 1: Moran's I Decay Curves (faceted by city)
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
for idx, (ax, city) in enumerate(zip(axes, CITIES)):
    city_df = morans_df[morans_df["city"] == city]
    for feature in sorted(city_df["feature"].unique()):
        subset = city_df[city_df["feature"] == feature]
        ax.plot(subset["distance_m"], subset["morans_i"], marker="o", label=feature)

    decay_values = decay_df[decay_df["city"] == city]["decay_distance_m"]
    ax.axhline(0.05, color="black", linestyle="--", linewidth=1, label="I=0.05")
    ax.axvline(recommended_block_size, color="red", linestyle="--", linewidth=1.5, label="Recommended")
    ax.axvspan(decay_values.min(), decay_values.max(), color="grey", alpha=0.2)

    ax.set_title(f"Moran's I Decay - {city.title()}", fontsize=14, fontweight="bold")
    ax.set_xlabel("Distance (m)", fontsize=12)
    ax.set_ylabel("Moran's I", fontsize=12)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, fontsize=10)
fig.tight_layout(rect=[0, 0.08, 1, 1])
save_figure(fig, FIGURES_DIR / "morans_i_decay_curves.png")

# Plot 2: Spatial Autocorrelation Heatmap (faceted by city)
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
for idx, (ax, city) in enumerate(zip(axes, CITIES)):
    city_df = morans_df[morans_df["city"] == city]
    pivot = city_df.pivot(index="feature", columns="distance_m", values="morans_i")
    sig = city_df.pivot(index="feature", columns="distance_m", values="significant")

    annot = pivot.copy().astype(str)
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            val = pivot.iloc[i, j]
            star = "*" if sig.iloc[i, j] else ""
            annot.iloc[i, j] = f"{val:.2f}{star}"

    sns.heatmap(
        pivot,
        cmap="RdBu_r",
        center=0,
        annot=annot,
        fmt="",
        ax=ax,
        cbar=idx == 0,
    )
    ax.set_title(f"Spatial Autocorrelation - {city.title()}", fontsize=14, fontweight="bold")
    ax.set_xlabel("Distance (m)", fontsize=12)
    ax.set_ylabel("Feature", fontsize=12)

fig.tight_layout()
save_figure(fig, FIGURES_DIR / "spatial_autocorrelation_heatmap.png")

# Plot 3: Decay Distance Summary
fig, ax = plt.subplots(figsize=PUBLICATION_STYLE["figsize"])
ax.bar(
    summary_df["feature"],
    summary_df["decay_mean"],
    yerr=summary_df["decay_std"].fillna(0),
    capsize=4,
)
ax.axhline(max_decay_distance, color="red", linestyle="--", label="Max decay")
ax.axhline(mean_plus_std, color="blue", linestyle=":", label="Mean + 1σ")
ax.axhline(recommended_block_size, color="black", linestyle="-.", label="Recommended block size")
ax.set_title("Decay Distance Summary", fontsize=14, fontweight="bold")
ax.set_xlabel("Feature", fontsize=12)
ax.set_ylabel("Decay Distance (m)", fontsize=12)
ax.legend(fontsize=10)
plt.xticks(rotation=45, ha="right")
fig.tight_layout()
save_figure(fig, FIGURES_DIR / "decay_distance_summary.png")

# Plot 4: Block Overlay Maps (per city)
for city in CITIES:
    grid = block_stats[city]["grid"]
    city_gdf = gpd.read_file(INPUT_DIR / f"trees_clean_{city}.gpkg")
    boundary = city_gdf.unary_union.convex_hull

    fig, ax = plt.subplots(figsize=PUBLICATION_STYLE["figsize"])
    gpd.GeoSeries(boundary, crs=city_gdf.crs).boundary.plot(ax=ax, color="black", linewidth=1)
    grid.plot(
        ax=ax,
        column="tree_count",
        cmap="viridis",
        legend=True,
        legend_kwds={"label": "Trees per block"},
        alpha=0.6,
    )
    city_gdf.plot(ax=ax, markersize=2, color="white", alpha=0.6)

    ax.set_title(
        f"Spatial Block Structure ({recommended_block_size}m × {recommended_block_size}m) - {city.title()}",
        fontsize=14,
        fontweight="bold",
    )
    ax.set_axis_off()

    save_figure(fig, FIGURES_DIR / f"block_overlay_map_{city}.png")

# Plot 5: Block Size Sensitivity Analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=False)

for idx, (ax, city) in enumerate(zip(axes, CITIES)):
    ax.plot(BLOCK_SIZE_CANDIDATES, block_counts[city], marker="o", label="Block count")
    ax2 = ax.twinx()
    ax2.plot(BLOCK_SIZE_CANDIDATES, residual_autocorr[city], color="tab:red", marker="s", label="Residual Moran's I")

    ax.axvline(recommended_block_size, color="black", linestyle="--", linewidth=1.5)
    ax.set_title(f"Block Size Sensitivity - {city.title()}", fontsize=14, fontweight="bold")
    ax.set_xlabel("Block size (m)", fontsize=12)
    ax.set_ylabel("Number of blocks", fontsize=12)
    ax2.set_ylabel("Residual Moran's I", fontsize=12, color="tab:red")

fig.tight_layout()
save_figure(fig, FIGURES_DIR / "block_size_sensitivity.png")

log.end_step(status="success")


In [ ]:
# ============================================================
# SECTION F: Save Configuration JSON
# ============================================================

log.start_step("Save JSON Config")


# Compute residual autocorrelation at recommended block size
def residual_autocorr_at_size(city: str) -> float:
    city_df = morans_df[morans_df["city"] == city]
    features = sorted(city_df["feature"].unique())
    values = [interpolate_moran(city_df, feat, recommended_block_size) for feat in features]
    return float(np.mean(values))

residual_at_recommended = float(np.mean([
    residual_autocorr_at_size("berlin"),
    residual_autocorr_at_size("leipzig"),
]))

created_ts = datetime.now(timezone.utc).replace(microsecond=0).isoformat()

output = {
    "version": "1.0",
    "created": created_ts,
    "recommended_block_size_m": int(recommended_block_size),
    "justification": (
        f"{recommended_block_size}m exceeds maximum autocorrelation decay distance "
        f"({max_decay_distance:.0f}m) with {safety_buffer_m}m safety buffer"
    ),
    "analysis_details": {
        "decay_distances": {
            row["feature"]: float(row["decay_max"]) for _, row in summary_df.iterrows()
        },
        "max_decay_distance": float(max_decay_distance),
        "aggregation_method": "maximum (conservative)",
        "safety_buffer_m": int(safety_buffer_m),
        "alternative_methods": {
            "mean_plus_std": float(mean_plus_std),
            "proportional_buffer_20pct": float(proportional_buffer_20pct),
        },
    },
    "distance_lags_tested": DISTANCE_LAGS,
    "features_analyzed": sorted(summary_df["feature"].tolist()),
    "validation": {
        "berlin_decay_distance": float(city_decay["berlin"]),
        "leipzig_decay_distance": float(city_decay["leipzig"]),
        "cross_city_consistency": (
            "high (difference < 50m)" if cross_city_diff <= 50 else "low (difference > 50m)"
        ),
        "berlin_sufficient_blocks": bool(block_stats["berlin"]["feasible"]),
        "leipzig_sufficient_blocks": bool(block_stats["leipzig"]["feasible"]),
        "block_size_exceeds_decay": bool(recommended_block_size >= max_decay_distance),
        "residual_autocorrelation_at_block_size": residual_at_recommended,
    },
    "block_counts": {
        "berlin": {
            "estimated_blocks": int(block_stats["berlin"]["blocks"]),
            "estimated_trees_per_block": float(block_stats["berlin"]["avg_trees_per_block"]),
            "feasible": bool(block_stats["berlin"]["feasible"]),
        },
        "leipzig": {
            "estimated_blocks": int(block_stats["leipzig"]["blocks"]),
            "estimated_trees_per_block": float(block_stats["leipzig"]["avg_trees_per_block"]),
            "feasible": bool(block_stats["leipzig"]["feasible"]),
        },
    },
    "sensitivity_analysis": {
        "alternative_block_sizes_tested": BLOCK_SIZE_CANDIDATES,
        "trade_offs": (
            f"{recommended_block_size}m provides the balance between spatial independence "
            "and sufficient block counts."
        ),
    },
}

output_path = METADATA_DIR / "spatial_autocorrelation.json"
output_path.write_text(json.dumps(output, indent=2), encoding="utf-8")
print(f"Saved JSON: {output_path}")

log.end_step(status="success")


In [ ]:
# ============================================================
# SUMMARY & MANUAL SYNC REQUIRED
# ============================================================

print("\n" + "=" * 70)
print("⚠️  MANUAL SYNC REQUIRED:")
print("=" * 70)
print(f"RECOMMENDED BLOCK SIZE: {recommended_block_size}m")
print()
print(f"1. JSON config: {METADATA_DIR / 'spatial_autocorrelation.json'}")
print(f"2. Plots: {FIGURES_DIR}")
print()
print("NEXT STEPS:")
print("1. Copy JSON from Drive to local repo:")
print("   cp ~/Google\\ Drive/.../spatial_autocorrelation.json outputs/phase_2/metadata/")
print("2. Copy plots from Drive to local repo:")
print("   cp ~/Google\\ Drive/.../exp_05_spatial/*.png outputs/phase_2/figures/exp_05_spatial/")
print("3. Review outputs")
print("4. Commit to Git:")
print("   git add outputs/phase_2/metadata/spatial_autocorrelation.json")
print("   git add outputs/phase_2/figures/exp_05_spatial/")
print("   git commit -m 'feat(phase-2): determine optimal spatial block size via Moran I analysis'")
print("5. Push to GitHub")
print("6. Next notebook (02c) will load the JSON automatically")
print("7. Proceed to Task 11 or 12")
print("=" * 70)